# SpanBERT Emotion-Cause QA


In [ ]:
# CELL 1: SETUP & MOUNT DRIVE
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
drive.mount('/content/drive')

import torch
import gc
import json
import shutil
import warnings
warnings.filterwarnings('ignore')

gc.collect()
torch.cuda.empty_cache()

PROJECT_ROOT            = "/content/drive/MyDrive/NLP_Project"
PROCESSED_DATA_DIR      = f"{PROJECT_ROOT}/processed_data"
SPANBERT_MODELS_DIR     = f"{PROJECT_ROOT}/models/spanbert_cause_qa"
SPANBERT_CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints/spanbert_qa"
RAW_DATASET_PATH        = f"{PROCESSED_DATA_DIR}/cause_qa_dataset"
TOKENIZED_DATASET_PATH  = f"{PROCESSED_DATA_DIR}/cause_qa_tokenized"

for dir_path in [PROCESSED_DATA_DIR, SPANBERT_MODELS_DIR, SPANBERT_CHECKPOINT_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print('='*70)
print(' Setup Complete')
print('='*70)
print(f"GPU Available : {torch.cuda.is_available()}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# CELL 2: CONFIGURATION
class SpanBERTQAConfig:
    RANDOM_SEED             = 42
    CONTEXT_WINDOW          = 3
    VALIDATION_SPLIT        = 0.1
    MAX_LENGTH              = 384
    DOC_STRIDE              = 128
    BATCH_SIZE              = 4
    GRADIENT_ACCUMULATION_STEPS = 8
    LEARNING_RATE           = 2e-5
    NUM_EPOCHS              = 3
    WARMUP_STEPS            = 100
    WEIGHT_DECAY            = 0.01
    SPANBERT_MODEL          = "SpanBERT/spanbert-base-cased"
    OUTPUT_DIR              = SPANBERT_MODELS_DIR
    CHECKPOINT_DIR          = SPANBERT_CHECKPOINT_DIR
    RAW_DATASET_PATH        = RAW_DATASET_PATH
    TOKENIZED_DATASET_PATH  = TOKENIZED_DATASET_PATH

print(' Configuration ready')

In [ ]:
# CELL 3: PREPROCESSING ECF 2.0 DATA
import os
import shutil
import collections
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

print('='*70)
print('STEP 1: PREPROCESSING ECF 2.0 DATA')
print('='*70)

print('\n1. Loading ECF 2.0 data...')
data_path = f'{PROJECT_ROOT}/data/Subtask_1_train.json'
with open(data_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
print(f"    Loaded {len(raw_data)} conversations")

print('\n2. Processing emotion-cause_pairs...')

qa_data = []
stats = {
    'conversations':        len(raw_data),
    'total_utterances':     0,
    'total_pairs':          0,
    'pairs_processed':      0,
    'cause_text_found':     0,
    'cause_text_not_found': 0,
}

for conv_idx, conversation in enumerate(raw_data):
    utterances          = conversation.get('conversation', [])
    emotion_cause_pairs = conversation.get('emotion-cause_pairs', [])

    stats['total_utterances'] += len(utterances)
    stats['total_pairs']      += len(emotion_cause_pairs)

    # Build lookup dicts — never mutate original utt dicts
    utt_map     = {}
    utt_idx_map = {}
    for i, utt in enumerate(utterances):
        uid = utt.get('utterance_ID')
        utt_map[uid]     = utt
        utt_idx_map[uid] = i

    for pair in emotion_cause_pairs:
        stats['pairs_processed'] += 1

        emotion_part = pair[0]   # e.g. "3_surprise"
        cause_part   = pair[1]   # e.g. "1_I realize I am totally naked ."

        # --- Parse emotion ---
        e_parts = emotion_part.split('_', 1)
        if len(e_parts) != 2:
            continue
        try:
            emotion_utt_id = int(e_parts[0])
            emotion        = e_parts[1].lower()
        except:
            continue

        if emotion_utt_id not in utt_map:
            continue

        emotion_text = utt_map[emotion_utt_id].get('text', '').strip()
        if not emotion_text:
            continue

        # --- Parse cause ---
        # CRITICAL FIX: use the SPAN TEXT from the pair (c_parts[1]),
        # NOT the full utterance text from utt_map.
        # e.g. pair gives "1_I realize I am totally naked ."
        #   → cause_source_utt_id = 1
        #   → cause_text = "I realize I am totally naked ."  ← the actual span
        # The full utterance 1 text is much longer — we only want this span.
        cause_source_utt_id = None
        c_parts = cause_part.split('_', 1)
        if len(c_parts) == 2:
            try:
                cause_source_utt_id = int(c_parts[0])
                cause_text = c_parts[1].strip()   # ← span text from the pair
            except:
                cause_source_utt_id = None
                cause_text = cause_part.strip()
        else:
            cause_text = cause_part.strip()

        if not cause_text:
            continue

        # --- Build full_context spanning cause utt → emotion utt ---
        # Context = "Speaker: full_utterance_text | Speaker: ... | Speaker: emotion_text"
        # The cause_text (span) will be findable inside this context because
        # the span is a substring of the full utterance text.
        if (cause_source_utt_id is not None
                and cause_source_utt_id in utt_idx_map
                and emotion_utt_id in utt_idx_map):

            span_start = min(utt_idx_map[cause_source_utt_id], utt_idx_map[emotion_utt_id])
            span_end   = max(utt_idx_map[cause_source_utt_id], utt_idx_map[emotion_utt_id])

            context_parts = []
            for u in utterances[span_start : span_end + 1]:
                spk = u.get('speaker', 'Unknown')
                txt = u.get('text', '')
                context_parts.append(f"{spk}: {txt}")
            full_context = ' | '.join(context_parts)

        else:
            # Fallback: just use the emotion utterance text
            full_context = f"{utt_map[emotion_utt_id].get('speaker','Unknown')}: {emotion_text}"

        # --- Find answer position ---
        # cause_text is a span like "I realize I am totally naked ."
        # full_context is like "Chandler: Alright , so ... I realize I am totally naked . | ..."
        # find() will locate cause_text correctly inside the full utterance text
        answer_start = full_context.find(cause_text)

        if answer_start == -1:
            stats['cause_text_not_found'] += 1
            continue

        # Sanity check: verify slice matches
        extracted = full_context[answer_start : answer_start + len(cause_text)]
        if extracted != cause_text:
            stats['cause_text_not_found'] += 1
            continue

        stats['cause_text_found'] += 1

        # Unique ID per pair
        qa_id = f"conv_{conv_idx}_emotion_{emotion_utt_id}_pair_{stats['pairs_processed']}"

        qa_data.append({
            'id':       qa_id,
            'title':    f'emotion_{emotion}',
            'context':  full_context,
            'question': f'What caused the {emotion}?',
            'answers':  {
                'text':         [cause_text],
                'answer_start': [answer_start]
            },
            'emotion':   emotion,
            'has_cause': True,
        })

print(f"\n   Statistics:")
print(f"   - Total pairs        : {stats['total_pairs']}")
print(f"   - Cause text found   : {stats['cause_text_found']}")
print(f"   - Cause text missing : {stats['cause_text_not_found']}")
print(f"   - QA examples built  : {len(qa_data)}")

# Verify answer positions are well spread (not all the same)
print('\n   Verifying answer_start distribution...')
starts  = [d['answers']['answer_start'][0] for d in qa_data]
top5    = collections.Counter(starts).most_common(5)
unique  = len(set(starts))
print(f"   Top 5 most common positions : {top5}")
print(f"   Unique positions            : {unique} / {len(starts)}")
if unique < 10:
    print("    WARNING: nearly all answers start at the same position!")
else:
    print("    Positions are well distributed — good!")

# Show a sample to verify manually
print('\n   Sample QA examples:')
for ex in qa_data[:3]:
    ans  = ex['answers']['text'][0]
    pos  = ex['answers']['answer_start'][0]
    ctx  = ex['context']
    extracted = ctx[pos : pos + len(ans)]
    match = '' if extracted == ans else '❌'
    print(f"   {match} Q: {ex['question']}")
    print(f"      Context  : {ctx[:80]}...")
    print(f"      Answer   : '{ans}'  @ char {pos}")
    print(f"      Extracted: '{extracted}'")
    print()

print('\n3. Creating train/val split...')
all_ids = [d['id'] for d in qa_data]
train_ids, val_ids = train_test_split(
    all_ids,
    test_size=SpanBERTQAConfig.VALIDATION_SPLIT,
    random_state=SpanBERTQAConfig.RANDOM_SEED,
)
train_ids_set = set(train_ids)
val_ids_set   = set(val_ids)
train_data = [d for d in qa_data if d['id'] in train_ids_set]
val_data   = [d for d in qa_data if d['id'] in val_ids_set]
print(f"    Train : {len(train_data)}")
print(f"    Val   : {len(val_data)}")
assert len(val_data) > 0, ' Validation set is empty!'

print('\n4. Creating HuggingFace datasets...')
def to_hf(data):
    return Dataset.from_dict({
        'id':       [d['id']       for d in data],
        'title':    [d['title']    for d in data],
        'context':  [d['context']  for d in data],
        'question': [d['question'] for d in data],
        'answers':  [d['answers']  for d in data],
    })

raw_dataset = DatasetDict({
    'train':      to_hf(train_data),
    'validation': to_hf(val_data),
})

# Wipe stale cache before saving
print('\n5. Saving raw dataset to Drive...')
if os.path.exists(RAW_DATASET_PATH):
    shutil.rmtree(RAW_DATASET_PATH)
    print("     Cleared stale raw cache")
if os.path.exists(TOKENIZED_DATASET_PATH):
    shutil.rmtree(TOKENIZED_DATASET_PATH)
    print("     Cleared stale tokenized cache")
raw_dataset.save_to_disk(RAW_DATASET_PATH)

print('\n' + '='*70)
print(' PREPROCESSING COMPLETE!')
print('='*70)
print(f"Saved to : {RAW_DATASET_PATH}")
print(f"Train    : {len(raw_dataset['train'])} | Val: {len(raw_dataset['validation'])}")
print('\n⚡ Now re-run: Cell 4 → Cell 6 → Cell 7')

In [ ]:
# CELL 4: TOKENIZATION
from transformers import AutoTokenizer
from datasets import DatasetDict

print('='*70)
print('STEP 2: TOKENIZATION')
print('='*70)

print('\n1. Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(SpanBERTQAConfig.SPANBERT_MODEL)
print('    Tokenizer loaded')

print('\n2. Loading raw dataset...')
raw_dataset = DatasetDict.load_from_disk(RAW_DATASET_PATH)

# Guard: confirm raw columns exist (catches stale tokenized data at this path)
expected_raw = {'id', 'title', 'context', 'question', 'answers'}
actual_cols  = set(raw_dataset['train'].column_names)
if not expected_raw.issubset(actual_cols):
    raise RuntimeError(
        f"\n Raw dataset has wrong columns!\n"
        f"   Expected : {expected_raw}\n"
        f"   Found    : {actual_cols}\n"
        f"   Fix      : Re-run Cell 3 (Preprocessing)."
    )
print(f"    Train : {len(raw_dataset['train'])} | Val : {len(raw_dataset['validation'])}")

print('\n3. Tokenizing...')

def tokenize_qa(examples):
    questions = [q.strip() for q in examples['question']]
    contexts  = examples['context']

    tokenized = tokenizer(
        questions,
        contexts,
        truncation='only_second',
        max_length=SpanBERTQAConfig.MAX_LENGTH,
        stride=SpanBERTQAConfig.DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length',
    )

    sample_mapping = tokenized.pop('overflow_to_sample_mapping')
    offset_mapping = tokenized.pop('offset_mapping')

    start_positions = []
    end_positions   = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        answers    = examples['answers'][sample_idx]

        if len(answers['answer_start']) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        answer_start_char = answers['answer_start'][0]
        answer_text       = answers['text'][0]
        answer_end_char   = answer_start_char + len(answer_text)

        sequence_ids  = tokenized.sequence_ids(i)
        context_start = None
        context_end   = None
        for idx, sid in enumerate(sequence_ids):
            if sid == 1:
                if context_start is None:
                    context_start = idx
                context_end = idx

        if context_start is None:
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Find start token
        token_start_index = context_start
        found_start = False
        while token_start_index <= context_end:
            if offsets[token_start_index][0] <= answer_start_char < offsets[token_start_index][1]:
                found_start = True
                break
            token_start_index += 1

        if not found_start:
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Find end token — use (answer_end_char - 1) for correct boundary
        token_end_index = context_end
        found_end = False
        while token_end_index >= context_start:
            if offsets[token_end_index][0] <= answer_end_char - 1 < offsets[token_end_index][1]:
                found_end = True
                break
            token_end_index -= 1

        if not found_end or token_start_index > token_end_index:
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_positions.append(token_start_index)
            end_positions.append(token_end_index)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions']   = end_positions
    return tokenized

# Wipe stale tokenized cache before saving fresh one
if os.path.exists(TOKENIZED_DATASET_PATH):
    shutil.rmtree(TOKENIZED_DATASET_PATH)
    print('   🗑️  Cleared stale tokenized cache')

tokenized_dataset = raw_dataset.map(
    tokenize_qa,
    batched=True,
    remove_columns=['id', 'title', 'context', 'question', 'answers'],
)
tokenized_dataset.set_format('torch')

# Save tokenized to its OWN path (never overwrites raw dataset)
tokenized_dataset.save_to_disk(TOKENIZED_DATASET_PATH)
print('    Tokenization complete and saved!')

print('\n4. Verifying positions...')
valid = zero = invalid = 0
n_check = min(200, len(tokenized_dataset['train']))
for idx in range(n_check):
    ex    = tokenized_dataset['train'][idx]
    start = ex['start_positions'].item()
    end   = ex['end_positions'].item()
    n_tok = len(ex['input_ids'])
    if start == 0 and end == 0:
        zero += 1
    elif 0 < start < n_tok and 0 < end < n_tok and start <= end:
        valid += 1
    else:
        invalid += 1

print(f"    Valid   : {valid}")
print(f"     Zero    : {zero} (truncated/unanswerable)")
print(f"    Invalid : {invalid}")
if invalid > 5:
    print('    Too many invalid — re-check tokenize_qa()')
if zero > n_check * 0.5:
    print('     >50% zeros — consider increasing MAX_LENGTH')

print('\n' + '='*70)
print(' TOKENIZATION COMPLETE!')
print('='*70)
print(f"Train : {len(tokenized_dataset['train'])} | Val : {len(tokenized_dataset['validation'])}")

In [ ]:
# CELL 5: TRAINING SETUP (using HuggingFace Trainer)
from transformers import (
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from datasets import DatasetDict

print('='*70)
print('STEP 3: TRAINING SETUP')
print('='*70)

# Safe reload in case runtime restarted
if 'tokenized_dataset' not in dir() or tokenized_dataset is None:
    print('Reloading tokenized_dataset from disk...')
    tokenized_dataset = DatasetDict.load_from_disk(TOKENIZED_DATASET_PATH)
    tokenized_dataset.set_format('torch')
    print(f"    Reloaded: Train={len(tokenized_dataset['train'])} | Val={len(tokenized_dataset['validation'])}")

print('\n1. Loading SpanBERT model...')
model = AutoModelForQuestionAnswering.from_pretrained(
    SpanBERTQAConfig.SPANBERT_MODEL
).to('cuda')
print('    Model loaded on GPU')

print('\n2. Configuring training...')
training_args = TrainingArguments(
    output_dir=SpanBERTQAConfig.CHECKPOINT_DIR,
    num_train_epochs=SpanBERTQAConfig.NUM_EPOCHS,
    per_device_train_batch_size=SpanBERTQAConfig.BATCH_SIZE,
    per_device_eval_batch_size=SpanBERTQAConfig.BATCH_SIZE,
    gradient_accumulation_steps=SpanBERTQAConfig.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=SpanBERTQAConfig.LEARNING_RATE,
    warmup_steps=SpanBERTQAConfig.WARMUP_STEPS,
    weight_decay=SpanBERTQAConfig.WEIGHT_DECAY,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=False,
    report_to='none',
)

print('\n3. Creating Trainer...')
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=default_data_collator,
)
print('    Trainer ready')
print('\n' + '='*70)

In [ ]:
# CELL 6: RELOAD FRESH MODEL + SANITY CHECK
# Run this cell instead of Cell 5 if you want a custom training loop
# or if you hit NaN loss and need to reset the model completely.

import torch
import torch.nn as nn
import gc
import math
import os
import shutil
from torch.utils.data import DataLoader
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, default_data_collator
from datasets import DatasetDict

gc.collect()
torch.cuda.empty_cache()

# ── STEP 0: Reload all variables from disk (safe after any runtime restart)
print('='*70)
print('STEP 0: Reloading variables from disk')
print('='*70)

tokenizer = AutoTokenizer.from_pretrained(SpanBERTQAConfig.SPANBERT_MODEL)
print(' Tokenizer loaded')

# ── Load or rebuild tokenized dataset ─────────────────────────────────
print('\nLoading tokenized dataset...')

def tokenize_qa_fn(examples, tok, max_len, stride):
    """Standalone tokenizer — used only if dataset needs rebuilding."""
    questions = [q.strip() for q in examples['question']]
    tokenized = tok(
        questions, examples['context'],
        truncation='only_second', max_length=max_len, stride=stride,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding='max_length',
    )
    sample_mapping = tokenized.pop('overflow_to_sample_mapping')
    offset_mapping = tokenized.pop('offset_mapping')
    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_mapping):
        answers = examples['answers'][sample_mapping[i]]
        if len(answers['answer_start']) == 0:
            start_positions.append(0); end_positions.append(0); continue

        asc = answers['answer_start'][0]
        aec = asc + len(answers['text'][0])
        sids = tokenized.sequence_ids(i)
        cs = ce = None
        for idx, sid in enumerate(sids):
            if sid == 1:
                cs = cs if cs is not None else idx
                ce = idx

        if cs is None:
            start_positions.append(0); end_positions.append(0); continue

        ts = cs
        found_s = False
        while ts <= ce:
            if offsets[ts][0] <= asc < offsets[ts][1]: found_s = True; break
            ts += 1

        te = ce
        found_e = False
        while te >= cs:
            if offsets[te][0] <= aec - 1 < offsets[te][1]: found_e = True; break
            te -= 1

        if not found_s or not found_e or ts > te:
            start_positions.append(0); end_positions.append(0)
        else:
            start_positions.append(ts); end_positions.append(te)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions']   = end_positions
    return tokenized

tok_path = SpanBERTQAConfig.TOKENIZED_DATASET_PATH
raw_path = SpanBERTQAConfig.RAW_DATASET_PATH

if os.path.exists(f"{tok_path}/dataset_dict.json"):
    tokenized_dataset = DatasetDict.load_from_disk(tok_path)
    tokenized_dataset.set_format('torch')
    # Verify it actually has tensor columns
    if 'input_ids' not in tokenized_dataset['train'].column_names:
        raise RuntimeError(' Tokenized dataset has wrong columns — re-run Cell 4.')
    print(f'    Loaded from disk')
elif os.path.exists(f"{raw_path}/dataset_dict.json"):
    print('   Tokenized not found — rebuilding from raw dataset...')
    raw_dataset = DatasetDict.load_from_disk(raw_path)
    if 'context' not in raw_dataset['train'].column_names:
        raise RuntimeError(
            ' Raw dataset also has wrong columns.\n'
            '   Both datasets are corrupted. Re-run Cell 3 (Preprocessing) then Cell 4 (Tokenization).'
        )
    print(f'   Raw loaded: Train={len(raw_dataset["train"])} | Val={len(raw_dataset["validation"])}')
    print('   Tokenizing... (1-2 mins)')
    import functools
    tokenized_dataset = raw_dataset.map(
        functools.partial(tokenize_qa_fn, tok=tokenizer,
                          max_len=SpanBERTQAConfig.MAX_LENGTH,
                          stride=SpanBERTQAConfig.DOC_STRIDE),
        batched=True,
        remove_columns=['id', 'title', 'context', 'question', 'answers'],
    )
    tokenized_dataset.set_format('torch')
    if os.path.exists(tok_path): shutil.rmtree(tok_path)
    tokenized_dataset.save_to_disk(tok_path)
    print('    Rebuilt and saved')
else:
    raise FileNotFoundError(
        '\n Both datasets are missing from disk!\n'
        '   You must re-run Cell 3 (Preprocessing) then Cell 4 (Tokenization) first.'
    )

print(f'   Train : {len(tokenized_dataset["train"])} | Val : {len(tokenized_dataset["validation"])}')

# ── STEP 1: Load fresh SpanBERT model ─────────────────────────────────
print('\n' + '='*70)
print('STEP 1: Loading fresh SpanBERT model')
print('='*70)
print('NOTE: MISSING qa_outputs.bias/weight is EXPECTED — not an error.')
print('      SpanBERT has no QA head; we initialise it below.')

model = AutoModelForQuestionAnswering.from_pretrained(
    SpanBERTQAConfig.SPANBERT_MODEL,
    ignore_mismatched_sizes=True,
)

# BERT-standard QA head init
nn.init.normal_(model.qa_outputs.weight, mean=0.0, std=0.02)
nn.init.zeros_(model.qa_outputs.bias)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
model.train()

print(f'\nQA head stats: mean={model.qa_outputs.weight.mean():.4f}, std={model.qa_outputs.weight.std():.4f}')
print(f' Model on {device}')

# ── STEP 2: Pre-training sanity check ─────────────────────────────────
print('\n' + '='*70)
print('STEP 2: Pre-training sanity check')
print('='*70)

loader = DataLoader(tokenized_dataset['train'], batch_size=1, collate_fn=default_data_collator)
batch  = next(iter(loader))
batch  = {k: v.to(device) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    outputs = model(**batch)

loss_val = outputs.loss.item()
print(f'   Loss         : {loss_val:.4f}  ← healthy range is 5–8')
print(f'   start_logits : {outputs.start_logits.min():.2f} to {outputs.start_logits.max():.2f}')
print(f'   end_logits   : {outputs.end_logits.min():.2f} to {outputs.end_logits.max():.2f}')

if math.isnan(loss_val) or math.isinf(loss_val):
    print('\n Loss is NaN/Inf — do NOT proceed to training!')
    print('   Check tokenized dataset for zero positions (run debug cell).')
elif loss_val > 20:
    print('\n  Loss is high — training can still proceed.')
else:
    print('\n Loss is healthy — safe to run training cell.')

model.train()

In [ ]:
# CELL 7: CUSTOM TRAINING LOOP (fully NaN-safe)
import torch
import torch.nn as nn
import gc
import tqdm
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, default_data_collator
from torch.utils.data import DataLoader

gc.collect()
torch.cuda.empty_cache()

print('='*70)
print(' STARTING TRAINING')
print('='*70)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Check for NaN in weights AND in tokenized data before starting ──
print('Pre-flight checks...')
nan_weights = [n for n, p in model.named_parameters() if torch.isnan(p).any()]
if nan_weights:
    raise RuntimeError(f' NaN in model weights: {nan_weights}\n   Re-run Cell 6.')
print('    Model weights clean')

# Check tokenized dataset for bad positions
n_check   = min(500, len(tokenized_dataset['train']))
zero_count = 0
for idx in range(n_check):
    ex = tokenized_dataset['train'][idx]
    s  = ex['start_positions'].item()
    e  = ex['end_positions'].item()
    if s == 0 and e == 0:
        zero_count += 1
zero_pct = zero_count / n_check * 100
print(f'   Zero positions in first {n_check} examples: {zero_count} ({zero_pct:.1f}%)')
if zero_pct > 80:
    raise RuntimeError(' >80% zero positions — re-run Cell 3 and Cell 4.')
print('    Data looks fine')

# Use fp32 explicitly + lower LR + higher warmup ──────────────
model = model.float()   # ensure no fp16 precision issues
model.train()

optimizer = AdamW(
    model.parameters(),
    lr=1e-5,            # FIX: halved from 2e-5 — more stable for fresh QA head
    eps=1e-8,
    weight_decay=0.01,
)

dataloader = DataLoader(
    tokenized_dataset['train'],
    batch_size=SpanBERTQAConfig.BATCH_SIZE,
    shuffle=True,
    collate_fn=default_data_collator,
)

num_epochs  = SpanBERTQAConfig.NUM_EPOCHS
total_steps = len(dataloader) * num_epochs
warmup_steps = int(0.1 * total_steps)   # FIX: 10% warmup (was 6%)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'\nTotal steps  : {total_steps}')
print(f'Learning rate: 1e-5')
print(f'Warmup steps : {warmup_steps} (10%)\n')

global_step  = 0
running_loss = 0.0
nan_count    = 0
best_loss    = float('inf')

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    progress_bar = tqdm.tqdm(dataloader, desc='Training')

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        # ── Check input batch for NaN/bad values before forward pass ──
        if torch.isnan(batch['input_ids'].float()).any():
            nan_count += 1
            continue

        # ──  Skip batches where ALL positions are zero (no signal) ──
        starts = batch['start_positions']
        ends   = batch['end_positions']
        if (starts == 0).all() and (ends == 0).all():
            continue   # nothing to learn from this batch

        outputs = model(**batch)
        loss    = outputs.loss

        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            optimizer.zero_grad()

            #  Detailed NaN diagnosis ──
            if nan_count <= 5:
                print(f'\n  NaN loss at global_step={global_step}')
                print(f'   starts : {starts.tolist()}')
                print(f'   ends   : {ends.tolist()}')
                # Check logits for explosion
                sl = outputs.start_logits
                el = outputs.end_logits
                print(f'   start_logits: min={sl.min():.2f} max={sl.max():.2f} '
                      f'nan={torch.isnan(sl).any()} inf={torch.isinf(sl).any()}')
                print(f'   end_logits  : min={el.min():.2f} max={el.max():.2f} '
                      f'nan={torch.isnan(el).any()} inf={torch.isinf(el).any()}')

                # Check if model weights have become NaN
                nan_w = [n for n, p in model.named_parameters() if torch.isnan(p).any()]
                if nan_w:
                    print(f'    NaN has infected weights: {nan_w[:3]}')
                    print(f'   Training cannot continue — re-run Cell 6 then Cell 7.')
                    break

            if nan_count > 50:
                print('\n Too many NaN batches — stopping.')
                break
            continue

        loss.backward()

        # ── FIX 6: Aggressive gradient clipping ──
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)  # tighter clip

        # ── FIX 7: Skip optimizer step if gradients exploded ──
        if torch.isnan(torch.tensor(grad_norm)) or grad_norm > 100:
            print(f'\n  Gradient explosion at step {global_step}: norm={grad_norm:.1f} — skipping')
            optimizer.zero_grad()
            nan_count += 1
            continue

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        global_step  += 1
        running_loss += loss.item()

        if global_step % 50 == 0:
            avg = running_loss / 50
            print(f'\nStep {global_step}: Loss={avg:.4f} | grad_norm={grad_norm:.3f} | NaN={nan_count}')
            running_loss = 0.0
            best_loss    = min(best_loss, avg)

        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'step': global_step,
            'nan':  nan_count,
        })
    else:
        print(f'\n Epoch {epoch+1} done | Steps={global_step} | NaN={nan_count}')
        continue
    break

print('\n' + '='*70)
print(f' TRAINING COMPLETE | Steps: {global_step} | NaN skipped: {nan_count}')
print(f'   Best avg loss: {best_loss:.4f}')
print('='*70)

if global_step > 0:
    model.save_pretrained(f"{SpanBERTQAConfig.OUTPUT_DIR}/final_model")
    tokenizer.save_pretrained(f"{SpanBERTQAConfig.OUTPUT_DIR}/final_model")
    print(' Model saved!')
else:
    print(' Model NOT saved — 0 successful steps.')

In [ ]:
# CELL 8: DIAGNOSE — Check for NaN in weights
import torch

nan_params = []
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        nan_params.append(name)
        print(f' NaN in: {name} | shape: {param.shape}')

if not nan_params:
    print(' No NaN in any weights')
else:
    print(f'\nTotal NaN params: {len(nan_params)}')

In [ ]:
# CELL 9: SAVE FINAL MODEL & EVALUATE
print('\n' + '='*70)
print('SAVING FINAL MODEL')
print('='*70)

# Safe tokenizer reload
try:
    tokenizer
except NameError:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(SpanBERTQAConfig.SPANBERT_MODEL)
    print('     Tokenizer reloaded')

final_model_path = f"{SpanBERTQAConfig.OUTPUT_DIR}/final_model"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f'\n Final model saved to: {final_model_path}')

print('\nEvaluating...')
eval_results = trainer.evaluate()
print(f'   eval_loss : {eval_results.get("eval_loss", "N/A"):.4f}')
print(f'   runtime   : {eval_results.get("eval_runtime", "N/A"):.1f}s')

print('\n' + '='*70)
print('ALL COMPLETE!')
print('='*70)

In [ ]:
# EMERGENCY: Re-save the trained model correctly
# Run this immediately — model is still in memory from Cell 7

import torch

# Verify model is trained (loss should be low on a sample batch)
from torch.utils.data import DataLoader
from transformers import default_data_collator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loader = DataLoader(tokenized_dataset['train'], batch_size=1, collate_fn=default_data_collator)
batch  = next(iter(loader))
batch  = {k: v.to(device) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    outputs = model(**batch)

loss_val = outputs.loss.item()
print(f"Current model loss on sample: {loss_val:.4f}")

if loss_val < 3.0:
    print(" This is the TRAINED model — saving now...")
    model.save_pretrained(f"{SpanBERTQAConfig.OUTPUT_DIR}/final_model")
    tokenizer.save_pretrained(f"{SpanBERTQAConfig.OUTPUT_DIR}/final_model")
    print(" Trained model saved successfully!")
else:
    print(" This model looks untrained — do NOT save, check what happened.")

model.train()

In [ ]:
# CELL 10: INFERENCE - TEST EXTRACTION
from transformers import pipeline

print('='*70)
print('INFERENCE — TEST EXTRACTION')
print('='*70)

model_path  = f"{SpanBERTQAConfig.OUTPUT_DIR}/final_model"
qa_pipeline = pipeline('question-answering', model=model_path, device=0)

test_examples = [
    {
        'context':  'Chandler: Alright , so I am back in high school , I am standing in the middle of the cafeteria , and I realize I am totally naked . | Chandler: Then I look down , and I realize there is a phone ... there .',
        'question': 'What caused the surprise?'
    },
    {
        'context':  'Joey: Instead of ... ? | Chandler: That is right .',
        'question': 'What caused the anger?'
    },
    {
        'context':  'She was very sad because her best friend moved to another city .',
        'question': 'What caused the sadness?'
    },
    {
        'context':  'I am so happy because I just got promoted at work today .',
        'question': 'What caused the happiness?'
    },
]

print('\nTesting cause extraction:\n')
for i, example in enumerate(test_examples, 1):
    result = qa_pipeline(
        question=example['question'],
        context=example['context']
    )
    print(f'Example {i}:')
    print(f'  Context    : {example["context"][:80]}...')
    print(f'  Question   : {example["question"]}')
    print(f'  Cause found: "{result["answer"]}"')
    print(f'  Confidence : {result["score"]:.3f}')
    print()

print('='*70)
print('✅ INFERENCE COMPLETE!')
print('='*70)

In [ ]:
# CROSS-CHECK: Reload model from Drive and re-evaluate
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline

print('='*70)
print('CROSS-CHECK: Reloading model from Drive')
print('='*70)

model_path = f"{SpanBERTQAConfig.OUTPUT_DIR}/final_model"

# Step 1: Reload tokenizer and model fresh from disk
print('\n1. Loading tokenizer from Drive...')
tokenizer_loaded = AutoTokenizer.from_pretrained(model_path)
print('    Tokenizer loaded')

print('\n2. Loading model from Drive...')
model_loaded = AutoModelForQuestionAnswering.from_pretrained(model_path)
model_loaded = model_loaded.to('cuda')
model_loaded.eval()
print('    Model loaded on GPU')

# Step 2: Verify it's the trained model (loss should be low)
print('\n3. Verifying model is trained (checking loss)...')
from torch.utils.data import DataLoader
from transformers import default_data_collator

loader = DataLoader(tokenized_dataset['train'], batch_size=1, collate_fn=default_data_collator)
batch  = next(iter(loader))
batch  = {k: v.to('cuda') for k, v in batch.items()}

with torch.no_grad():
    outputs = model_loaded(**batch)

loss_val = outputs.loss.item()
print(f'   Sample loss: {loss_val:.4f}  ← should be below 1.0')
if loss_val < 1.0:
    print('    Confirmed — this is the trained model')
else:
    print('    Loss too high — model may not have saved correctly')

# Step 3: Run inference with reloaded model
print('\n4. Running inference with reloaded model...')
print('='*70)

qa_pipeline_reloaded = pipeline(
    'question-answering',
    model=model_loaded,
    tokenizer=tokenizer_loaded,
    device=0,
)

test_examples = [
    {
        'context':  'Chandler: Alright , so I am back in high school , I am standing in the middle of the cafeteria , and I realize I am totally naked . | Chandler: Then I look down , and I realize there is a phone ... there .',
        'question': 'What caused the surprise?'
    },
    {
        'context':  'Joey: Instead of ... ? | Chandler: That is right .',
        'question': 'What caused the anger?'
    },
    {
        'context':  'She was very sad because her best friend moved to another city .',
        'question': 'What caused the sadness?'
    },
    {
        'context':  'I am so happy because I just got promoted at work today .',
        'question': 'What caused the happiness?'
    },
]

print('\nResults:\n')
all_correct = True
expected = [
    "I realize I am totally naked .",
    "Instead of ... ?",
    "her best friend moved to another city .",
    "I just got promoted at work today .",
]

for i, (example, exp) in enumerate(zip(test_examples, expected), 1):
    result = qa_pipeline_reloaded(
        question=example['question'],
        context=example['context'],
    )
    answer     = result['answer']
    confidence = result['score']
    match      = '' if exp.lower() in answer.lower() or answer.lower() in exp.lower() else '⚠️'
    if match == '':
        all_correct = False

    print(f'Example {i}: {match}')
    print(f'  Question   : {example["question"]}')
    print(f'  Expected   : "{exp}"')
    print(f'  Got        : "{answer}"')
    print(f'  Confidence : {confidence:.3f}')
    print()

print('='*70)
if all_correct:
    print(' CROSS-CHECK PASSED — reloaded model matches expected outputs!')
else:
    print('  Some answers differ from expected — review above.')
print('='*70)